In [ ]:
# These are the libraries you may use for this assignment.
# If you would like to use any other libraries, email james@uwaterloo.ca for permission first.
# Generative AI tools (including coding assistants) are not permitted for this assignment.

from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

## CFM 101 — Group Assignment 2026
### Robo-Advising Challenge: Two Clients, Two Strategies

**Team Number:** 5  
**Team Member Names (three or four):**  

Enter the names of all participating members below (one name per line): 

John Mandalas

Mohammadreza Babaeizadeh

Chloe Pang

Pranav Sudev

**Submission:** Friday, November 27, 2026, at 9:00 AM (Eastern), via Dropbox.  
**Submit this completed Jupyter notebook only.**

Your team must construct **both** portfolios from the supplied `Tickers.csv` file:

- **Client A — Growth:** $\$1,000,000$ CAD; seek the highest expected return.
- **Client B — Low Risk:** $\$750,000$ CAD; seek the lowest portfolio volatility.

Both clients require 10–20 eligible stocks, at most 15% in any stock and at most 40% in any sector. Read the full assignment handout for all eligibility, fee, currency, and competition rules. Do not hardcode the selected stocks. Check that tickers are valid. The supplied ticker universe will permit feasible portfolios.

**No generative AI may be used in preparing this assignment.**

## Data preparation and shared functions

Read `Tickers.csv`, check ticker eligibility, retrieve the data needed for your strategies, and create any functions you will reuse for both clients. Use closing prices from **November 27, 2026** for the purchase calculations. All portfolio values and weights must be calculated in CAD, with purchase fees paid from the client budget.

## Variables

Global variables that can be configured.


In [ ]:
# Client Configurations
client_a_budget = 100000
client_b_budget = 750000

min_port_size = 10
max_port_size = 20

min_weight = 0.15
max_weight = 0.40

# Stock eligibility
csv_file_attach = "Tickers_Example.csv"
min_avg_d_vol = 5000

start_date = "2024-10-01"
end_date = "2025-10-01"

min_trade_d_per_m = 18

# Purchase and transaction fees in USD

fee_cap_per_stock = 2.15
fee_per_share = 0.001  

In [ ]:
# Gets all the list of tickers from the csv file and returns them as a pandas dataframe
# Parameters: file_name (str) - the name of the csv file containing the tickers
# Returns: pandas dataframe containing the tickers
def get_listof_tickers(file_name):
    listof_tickers = pd.read_csv(file_name, header=None) #first row info needs to be read.(Not header)
    return listof_tickers

# Normalizes the list of tickers to ensure they fit with the assignment criteria
# Parameters: tickers (pandas dataframe) - the dataframe containing the tickers to normalize
# Returns: pandas dataframe containing the normalized tickers
def normalize_listof_tickers(tickers):
    stock_data = []
    valid_exchanges = ["NMS", "NYQ", "ASE", "TOR", "VAN", "NEO"]# TBDTBDTBD

    # Get data needed for each ticker in the dataframe.
    for ticker in tickers[0]:
        try:
            stock = yf.Ticker(ticker)
            info = stock.info
            history = stock.history(start=start_date, end=end_date, auto_adjust=False) # Last trading date checked: 2026-09-30

            # Skip stocks without historical data
            if history.empty:
                continue
            # Check if the stock meets the basic requirements
            valid_stock = (info.get("quoteType") == "EQUITY" 
                           and info.get("currency") in ["CAD", "USD"] 
                           and info.get("exchange") in valid_exchanges)
           
            # Counting trading days in each calendar month
            month_counts = {}
            # Go through all dates and count trading days for each month
            for date in history.index:
                month = date.month
                
                if month in month_counts:
                    month_counts[month] += 1
                else:
                    month_counts[month] = 1

            lowest_trading_day = min(month_counts.values())

            # Calculate average daily trading volume
            total_volume = 0
            total_days = 0
            
            for date, row in history.iterrows():
                month = date.month
                # Only include months with at least 18 trading days
                if month_counts[month] >= min_trade_d_per_m: 
                    total_volume += row["Volume"]
                    total_days += 1
            
            if total_days > 0:
                average_volume = total_volume / total_days
            else:
                average_volume = 0

            # Creating a dataframe containing information needed in further functions
            if valid_stock and average_volume >= 5000:
                stock_data.append({
                    "Ticker": ticker,
                    "Stock's Name": info.get("longName"),
                    "Exchange": info.get("exchange"),
                    "Currency": info.get("currency"),
                    "Ticker Type": info.get("quoteType"),
                    "Sector": info.get("sector"),
                    "Monthly Trading Days": lowest_trading_day,
                    "Average Volume": average_volume
            })

        # Mark the stock as invalid if an error occurs
        except Exception:
            print("Not Valid Ticker:", ticker)
    return pd.DataFrame(stock_data)

# Returns: a dataframe cointaining the {TICKER}_CLOSE and {TICKER}_ADJ_CLOSE for each ticker in the list of tickers, for the given start and end dates
def create_dataframe(listof_tickers, start_date, end_date):
    stock_prices = pd.DataFrame()
    for ticker in listof_tickers["Ticker"]: # Change [0] to ["Ticker"] just to get the valid stocks price
        stock = yf.Ticker(ticker)
        history = stock.history(start=start_date, end=end_date,auto_adjust=False)

        if not history.empty:
            history.index = history.index.tz_localize(None)
            prices = history[["Close", "Adj Close"]].copy() #Add other categories to here
            prices.columns = [ticker + "_Close", ticker + "_Adj_Close"]
            stock_prices = pd.concat([stock_prices, prices], axis=1)

    return stock_prices

# Gets the USD to CAD exchange rate between two dates using yfinance
# Parameters: start_date (str) - the start date to get the exchange rate for in the format 'YYYY-MM-DD'
# Parameters: end_date (str) - the end date to get the exchange rate for in the format 'YYYY-MM-DD'
# Returns: float - the USD to CAD exchange rate for the given date
def get_USD_to_CAD_exchange_rate(start_date, end_date):
    # Get the USD to CAD exchange rate for the given date using yfinance
    exchange_rate = yf.download('CAD=X', start=start_date, end=end_date)
    return exchange_rate['Close']

# Buys a ticker and adds it to the portfolio, including fees
# Parameters: ticker_name (str) - the name of the ticker to buy, dollars (float) - the amount of money to spend on the ticker, portfolio_name (str) - the name of the portfolio to add the ticker to
# Returns: num_shares (float) - the number of shares bought
def buy_ticker(ticker_name, dollars, portfolio_name):
    # Buy the ticker, including fees, and add it to the portfolio (see assignment details for how to do this)
    # Update the given portfolio dataframe with the new ticker and number of shares bought
    fees = min(exchange_rate * 2.15, shares * exchange_rate * 0.001)
    return num_shares


# Calculates volatility of the whole portfolio over a given time period using the standard deviation of the returns
# Parameters: daily_returns (pandas series), the series that contains the daily returns of the portfolio over a given time period
# Returns: the volatility of the portfolio 
def calculate_volatility(daily_returns):
    return daily_returns.std()

# Calculate transaction fees for buying a ticker
# Parameters: shares (float), the number of shares to buy
# exchange_rate (float), CAD per US$1, the exchange rate for the given date
# Returns: the transaction fees for the stock purchase (in CAD)
def calculate_transaction_fees(shares, exchange_rate):
    fees = min(exchange_rate * 2.15, shares * exchange_rate * 0.001)
    return fees


"""
TO DO:

Volatility function
Get ticker price function
Beta calculation funtion
"""

# Calling ticker list
listof_tickers = get_listof_tickers(csv_file_attach)
display(listof_tickers)

# Calling valid ticker 
normalized_tickers = normalize_listof_tickers(listof_tickers)
display(normalized_tickers)

# Calling create_dataframe for Adj and Close price
stock_prices = create_dataframe(normalized_tickers, start_date, end_date)
display(stock_prices)

# Calling the exchange_rates USD CAD
exchange_rates = get_USD_to_CAD_exchange_rate('2020-01-01', '2020-12-31')

$AGN: possibly delisted; no timezone found
$BK: possibly delisted; no price data found  (1d 2020-01-01 -> 2020-12-31) (Yahoo error = "Data doesn't exist for startDate = 1577854800, endDate = 1609390800")
$CELG: possibly delisted; no timezone found
$MON: possibly delisted; no timezone found
$RTN: possibly delisted; no timezone found
[*********************100%***********************]  1 of 1 completed


## Client A — Growth

Construct a portfolio intended to achieve the **highest expected return**. Use your selected stocks and portfolio weights to create a DataFrame named `Portfolio_Final_A` with index **1 through n** and columns (in this order): `Ticker`, `Price`, `Currency`, `Shares`, `Value`, `Weight`. `Price` is in the stock’s quoted currency, `Value` is in CAD, and `Weight` is a fraction summing to 1.

### Client A — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

## Client B — Low Risk

Construct a portfolio intended to achieve the **lowest risk**, defined as the standard deviation of daily portfolio returns in CAD. Create `Portfolio_Final_B` using the same DataFrame column names, order, and index convention as for Client A.

### Client B — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

## Equal-weight comparisons

For **each client**, create an equal-weighted comparison portfolio holding **exactly the same stocks** as that client’s chosen portfolio, with starting weight `1/n` per stock. Over the same clearly identified historical period, calculate **historical return** and **volatility** for the chosen portfolio and its equal-weighted counterpart. Use consistent CAD valuations and weighting assumptions.

Display one concise comparison table per client, and interpret the results in the financial justifications above. No additional graph or out-of-sample test is required.

## Programmatic validation

Use Python (`assert` statements or a simple function are fine) to display whether **both** final portfolios pass each of the following checks:

- Selected tickers are valid and eligible.
- Number of stocks is between 10 and 20.
- No individual stock weight exceeds 15%.
- No single sector has more than 40% of portfolio value.
- Portfolio weights sum to 100%.
- Total stock purchases plus transaction fees do not exceed the applicable client budget.

Your code should run with the TAs’ ticker file without manually changing individual stock selections.

## Final portfolio outputs

Display `Portfolio_Final_A` and `Portfolio_Final_B`, along with the budget reconciliation (purchases plus fees) for each. Generate `Stocks_Group_XX_A.csv` and `Stocks_Group_XX_B.csv` (replace `XX` with your team number). Each CSV must contain **only** the columns `Ticker` and `Shares`. The notebook must **generate** these CSV files when run, but you submit **only the notebook** to Dropbox.

## Contribution and AI Declaration

**The following team members made a meaningful contribution to this assignment:**

Enter the names of all contributing members below (one name per line):

**To the best of our knowledge, no generative AI was used in preparing this submission.**

List only members who made a meaningful contribution. Contact the instructor before the deadline if someone should be omitted or if the team cannot make the declaration.